In [9]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import roc_auc_score, f1_score, matthews_corrcoef
from sklearn.utils.class_weight import compute_class_weight
from scipy.stats import pearsonr

# =========================
# SETTINGS
# =========================

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

BATCH_SIZE = 32
EPOCHS = 30
LR = 3e-4

LAMBDA_PATH = 1.0
LAMBDA_DISEASE = 1.2
LAMBDA_DDG = 0.5

EARLY_STOP_PATIENCE = 6

print("Using device:", DEVICE)

Using device: cuda


In [10]:
# =========================
# LOAD DATA
# =========================

X_train = torch.load(r"C:\SEM 4\patho\train_features.pt")
X_val   = torch.load(r"C:\SEM 4\patho\val_features.pt")
X_test  = torch.load(r"C:\SEM 4\patho\test_features.pt")

train_df = pd.read_csv(r"C:\SEM 4\patho\Processed\train.csv")
val_df   = pd.read_csv(r"C:\SEM 4\patho\Processed\val.csv")
test_df  = pd.read_csv(r"C:\SEM 4\patho\Processed\test.csv")

C:\Users\mbm54\AppData\Local\Temp\ipykernel_28112\1886834623.py:5: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  X_train = torch.load(r"C:\SEM 4\patho\train_features.pt")
C:

In [11]:
# =========================
# LABELS
# =========================

# Binary Pathogenicity
y_train_path = torch.tensor(
    train_df["ClinicalSignificance"].apply(lambda x: 1 if x=="Pathogenic" else 0).values,
    dtype=torch.float32)

y_val_path = torch.tensor(
    val_df["ClinicalSignificance"].apply(lambda x: 1 if x=="Pathogenic" else 0).values,
    dtype=torch.float32)

# Disease
disease_classes = sorted(train_df["Disease_Category"].unique())
disease_map = {d:i for i,d in enumerate(disease_classes)}

y_train_dis = torch.tensor(
    train_df["Disease_Category"].map(disease_map).values,
    dtype=torch.long)

y_val_dis = torch.tensor(
    val_df["Disease_Category"].map(disease_map).values,
    dtype=torch.long)

# dDG (delta delta G)
y_train_ddg = torch.tensor(train_df["ddG"].values, dtype=torch.float32)
y_val_ddg   = torch.tensor(val_df["ddG"].values, dtype=torch.float32)

train_mask = ~torch.isnan(y_train_ddg)
val_mask   = ~torch.isnan(y_val_ddg)

y_train_ddg[~train_mask] = 0
y_val_ddg[~val_mask] = 0

In [12]:
# =========================
# DATASET
# =========================

class MutationDataset(Dataset):
    def __init__(self, X, y_path, y_dis, y_ddg, mask):
        self.X = X
        self.y_path = y_path
        self.y_dis = y_dis
        self.y_ddg = y_ddg
        self.mask = mask

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return (
            self.X[idx],
            self.y_path[idx],
            self.y_dis[idx],
            self.y_ddg[idx],
            self.mask[idx]
        )

train_loader = DataLoader(
    MutationDataset(X_train, y_train_path, y_train_dis, y_train_ddg, train_mask),
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    MutationDataset(X_val, y_val_path, y_val_dis, y_val_ddg, val_mask),
    batch_size=BATCH_SIZE
)

In [13]:
# =========================
# MODEL
# =========================

class MultiTaskModel(nn.Module):
    def __init__(self, input_dim, n_disease):
        super().__init__()

        self.shared = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.3)
        )

        self.path_head = nn.Linear(256, 1)

        # Deeper disease head
        self.dis_head = nn.Sequential(
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, n_disease)
        )

        self.ddg_head = nn.Linear(256, 1)

    def forward(self, x):
        rep = self.shared(x)
        return (
            self.path_head(rep).squeeze(),
            self.dis_head(rep),
            self.ddg_head(rep).squeeze()
        )

model = MultiTaskModel(
    input_dim=X_train.shape[1],
    n_disease=len(disease_classes)
).to(DEVICE)

In [ ]:
# =========================
# LOSSES
# =========================

bce = nn.BCEWithLogitsLoss()

class MultiClassFocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=1.0):
        super(MultiClassFocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        return focal_loss.mean()

class_counts = np.bincount(y_train_dis.numpy())
smoothed_weights = 1.0 / np.sqrt(class_counts)
smoothed_weights = smoothed_weights / np.mean(smoothed_weights)
weights = torch.tensor(smoothed_weights, dtype=torch.float32).to(DEVICE)

ce = MultiClassFocalLoss(alpha=weights, gamma=1.0).to(DEVICE)
mse = nn.MSELoss()

optimizer = torch.optim.Adam(model.parameters(), lr=LR)

In [15]:
# =========================
# TRAINING LOOP
# =========================

best_score = 0
patience_counter = 0

for epoch in range(EPOCHS):

    model.train()
    total_loss = 0

    for X, y_path, y_dis, y_ddg, mask in train_loader:

        X = X.to(DEVICE)
        y_path = y_path.to(DEVICE)
        y_dis = y_dis.to(DEVICE)
        y_ddg = y_ddg.to(DEVICE)
        mask = mask.to(DEVICE)

        path_pred, dis_pred, ddg_pred = model(X)

        loss_path = bce(path_pred, y_path)
        loss_dis  = ce(dis_pred, y_dis)

        if mask.sum() > 0:
            loss_ddg = mse(ddg_pred[mask], y_ddg[mask])
        else:
            loss_ddg = torch.tensor(0.0).to(DEVICE)

        loss = (
            LAMBDA_PATH * loss_path +
            LAMBDA_DISEASE * loss_dis +
            LAMBDA_DDG * loss_ddg
        )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"\nEpoch {epoch+1} | Train Loss: {total_loss:.2f}")

    # =========================
    # VALIDATION
    # =========================

    model.eval()

    all_path_pred = []
    all_path_true = []
    all_dis_pred = []
    all_dis_true = []
    all_ddg_pred = []
    all_ddg_true = []

    with torch.no_grad():
        for X, y_path, y_dis, y_ddg, mask in val_loader:

            X = X.to(DEVICE)
            path_pred, dis_pred, ddg_pred = model(X)

            all_path_pred.extend(torch.sigmoid(path_pred).cpu().numpy())
            all_path_true.extend(y_path.numpy())

            all_dis_pred.extend(dis_pred.argmax(1).cpu().numpy())
            all_dis_true.extend(y_dis.numpy())

            all_ddg_pred.extend(ddg_pred.cpu().numpy())
            all_ddg_true.extend(y_ddg.numpy())

    auc = roc_auc_score(all_path_true, all_path_pred)
    disease_f1 = f1_score(all_dis_true, all_dis_pred, average="macro")
    r, _ = pearsonr(all_ddg_true, all_ddg_pred)

    composite_score = 0.5 * auc + 0.25 * disease_f1 + 0.25 * r

    print(f"AUC: {auc:.4f}")
    print(f"Disease Macro-F1: {disease_f1:.4f}")
    print(f"dDG Pearson r: {r:.4f}")
    print(f"Composite Score: {composite_score:.4f}")

    if composite_score > best_score:
        best_score = composite_score
        torch.save(model.state_dict(), "best_multitask_model.pt")
        print("Saved best composite model.")
        patience_counter = 0
    else:
        patience_counter += 1

    if patience_counter >= EARLY_STOP_PATIENCE:
        print("Early stopping triggered.")
        break

print("\nTraining complete.")


Epoch 1 | Train Loss: 558.66
AUC: 0.8274
Disease Macro-F1: 0.1526
dDG Pearson r: 0.6350
Composite Score: 0.6106
Saved best composite model.

Epoch 2 | Train Loss: 491.56
AUC: 0.8437
Disease Macro-F1: 0.1646
dDG Pearson r: 0.6726
Composite Score: 0.6311
Saved best composite model.

Epoch 3 | Train Loss: 460.29
AUC: 0.8440
Disease Macro-F1: 0.1835
dDG Pearson r: 0.6857
Composite Score: 0.6393
Saved best composite model.

Epoch 4 | Train Loss: 440.84
AUC: 0.8508
Disease Macro-F1: 0.2036
dDG Pearson r: 0.7223
Composite Score: 0.6569
Saved best composite model.

Epoch 5 | Train Loss: 423.99
AUC: 0.8551
Disease Macro-F1: 0.2157
dDG Pearson r: 0.7334
Composite Score: 0.6648
Saved best composite model.

Epoch 6 | Train Loss: 411.03
AUC: 0.8594
Disease Macro-F1: 0.2390
dDG Pearson r: 0.7351
Composite Score: 0.6733
Saved best composite model.

Epoch 7 | Train Loss: 397.54
AUC: 0.8621
Disease Macro-F1: 0.2436
dDG Pearson r: 0.7557
Composite Score: 0.6809
Saved best composite model.

Epoch 8 | Tr